# TP3 : consumer groups, offsets, rétention et compaction

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 14:05-15:00, après la démo du TP (14:00-14:05) |
| Compétences | C5 Produire, consommer, gérer les groupes et les offsets ; C6 Rétention et compaction |
| Slides de rappel | S11 (consumer groups), S12 (rééquilibrage et KIP-848), S29 (commits et `auto.offset.reset`), S31 (rétention et compaction) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule, avec trois terminaux |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- La version Markdown de ce TP utilise trois terminaux pour faire tourner plusieurs consommateurs. Ici, chaque consommateur est une **instance** qui tourne en arrière-plan dans le notebook (classe `Instance`, étape 6.3).
- Les fonctions `topic_offsets()` et `end_offsets()` reprennent le TP2 : elles sont fournies dans la cellule suivante.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

In [ ]:
import csv
import re
import threading
import time

from confluent_kafka import (OFFSET_BEGINNING, Consumer, ConsumerGroupState, ConsumerGroupTopicPartitions,
                             ConsumerGroupType, KafkaException, Producer, TopicPartition)
from confluent_kafka.admin import NewTopic, OffsetSpec


def topic_offsets(topic, spec):
    """{partition: offset} pour une spécification : OffsetSpec.earliest(), latest() ou for_timestamp(ms)."""
    admin = AdminClient(CONF)
    num_partitions = len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
    futures = admin.list_offsets({TopicPartition(topic, p): spec for p in range(num_partitions)})
    return {tp.partition: f.result().offset for tp, f in sorted(futures.items(), key=lambda x: x[0].partition)}


def end_offsets(topic):
    return topic_offsets(topic, OffsetSpec.latest())


print("Bootstrap INTERNAL :", BS)
print("orders :", end_offsets("orders"))

## 1. Objectifs

À la fin du TP, vous savez :

- consommer avec un consumer group et calculer son lag ;
- observer la répartition des partitions entre consommateurs et le rééquilibrage ;
- reconnaître un groupe utilisant le nouveau protocole (KIP-848) ;
- repositionner les offsets d'un groupe en sécurité : simulation, application, export ;
- faire supprimer des données par la rétention en temps ;
- configurer un topic compacté et supprimer une clé avec un tombstone.

## 2. Prérequis

- TP2 terminé : topic `orders` avec 6 partitions et 60 messages (voir la cellule précédente). Sinon, demandez au formateur le bloc de rattrapage du TP2.
- Jeu de données `lab/data/product-prices.txt` : 20 prix pour 4 produits `P-100` à `P-400`.
- Notions : slides S11, S29, S31.

## 3. Architecture utilisée

```
                       orders (6 partitions)
          p0    p1    p2    p3    p4    p5
           \     |     /     \     |     /
   groupe analytics : instance a (p0, p1, p2) + instance b (p3, p4, p5)
   groupe billing   : un consommateur, lancé puis arrêté (offsets conservés)

   Offsets validés : topic interne __consumer_offsets (50 partitions, compacté)
   Coordinateur du groupe : le leader de la partition de __consumer_offsets
   qui correspond au nom du groupe.
```

| Topic | Partitions | Configuration | Usage |
|---|---|---|---|
| `orders` | 6 | défaut | Groupes et offsets |
| `audit-short` | 1 | `retention.ms=60000`, `segment.ms=10000` | Rétention en temps |
| `product-prices` | 1 | `cleanup.policy=compact`, segments de 10 s | Compaction |

## 4. Contexte métier

Chez NovaShop, deux services lisent les commandes :

- **Facturation** (groupe `billing`) doit traiter chaque commande une fois, et pouvoir rejouer une période après un incident ;
- **Analytique** (groupe `analytics`) doit monter en charge en ajoutant des instances.

Le service Audit veut un journal court, et le service Catalogue un référentiel de prix qui ne garde que le dernier prix de chaque produit. L'état des groupes s'affiche en fin de notebook.

## 5. Énoncé

1. Faites consommer `orders` par la facturation, puis montrez le lag créé par de nouvelles commandes.
2. Faites travailler deux instances de l'analytique en parallèle, et observez ce qui se passe quand l'une s'arrête.
3. Repositionnez la facturation pour rejouer des commandes, sans jamais appliquer un changement que vous n'avez pas simulé.
4. Prouvez que la rétention supprime les vieux messages d'`audit-short` et que la compaction ne garde que le dernier prix de chaque produit.

## 6. Étapes

### 6.1 Consommer avec un groupe

Un consommateur qui appartient à un groupe valide (« commit ») ses offsets dans le topic interne `__consumer_offsets`. À son redémarrage, il reprend à ces offsets. `auto.offset.reset` ne sert que pour une partition où le groupe n'a encore rien validé : `earliest` lit alors depuis le début, `latest` (valeur par défaut) seulement les nouveaux messages.

**Exemple en ligne de commande** : les groupes existants.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-console-consumer.sh --bootstrap-server $BS --topic orders --group billing \
  --from-beginning --timeout-ms 10000 | wc -l
kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing
```

In [ ]:
# TODO 6.1 : consommer tout orders avec le groupe billing, puis calculer son lag
# 1. Complétez consume(group, topic) : elle lit jusqu'à ce qu'aucun message n'arrive pendant 5 s,
#    ferme le consommateur et renvoie la liste des messages lus.
#    consumer.consume(100, timeout=1) renvoie une liste de messages (vide si rien n'arrive en 1 s).
def consume(group, topic, **config):
    # earliest : un groupe sans offset validé lit depuis le début.
    # Les offsets sont validés automatiquement (enable.auto.commit), et une dernière fois par close().
    consumer = Consumer({"bootstrap.servers": BS, "group.id": group, "auto.offset.reset": "earliest",
                         "log_level": 4, **config})
    consumer.subscribe([topic])
    messages = []
    # à compléter
    consumer.close()
    return messages


# 2. Écrivez committed_offsets(group, topic) : {partition: offset valide}.
#    admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(group)]) renvoie {group: futur} ;
#    le résultat a un attribut topic_partitions : liste de TopicPartition (topic, partition, offset).
#    Une partition absente de la liste, ou d'offset négatif, n'a rien de validé : comptez 0.
def committed_offsets(group, topic):
    return {}


# 3. Écrivez lag(group, topic) : {partition: offset de fin - offset valide}, avec end_offsets().
def lag(group, topic):
    return {}


consumed = consume("billing", "orders")
print(len(consumed), "messages lus")
print("lag :", lag("billing", "orders"))

In [ ]:
# Vérification 6.1
admin = AdminClient(CONF)
assert len(consumed) == 60, "À revoir : 60 messages lus (%d)" % len(consumed)
print("OK  60 messages lus (%d)" % len(consumed))
assert lag("billing", "orders") == {p: 0 for p in range(6)}, "À revoir : lag nul sur les 6 partitions"
print("OK  lag nul sur les 6 partitions")
state = admin.describe_consumer_groups(["billing"])["billing"].result().state
assert state == ConsumerGroupState.EMPTY, "À revoir : groupe billing sans membre actif : %s" % state.name
print("OK  groupe billing sans membre actif : %s" % state.name)

**Résultat attendu** : 60 messages lus, en une dizaine de secondes. Le lag est nul partout et le groupe est `EMPTY` : il n'a plus de membre, mais ses offsets sont conservés.

### 6.2 Créer et résorber du lag

Le lag d'une partition est l'écart entre son offset de fin et l'offset validé du groupe : le nombre de messages qui restent à traiter.

Équivalent en ligne de commande de l'envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
for i in $(seq 1 10); do
  echo "C008|{\"order_id\":\"O-30$i\",\"customer\":\"C008\",\"amount\":10.0,\"city\":\"Lyon\"}"
done | kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|'
```

**Exécutez la cellule `TODO` une seule fois** : chaque exécution envoie 10 commandes.

In [ ]:
# TODO 6.2 (a) : envoyer 10 commandes du nouveau client C008, puis mesurer le lag de billing
# Le producteur a la même configuration qu'au TP2 : les clés sont rangées comme avec les outils Java.
producer = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4})
# Valeur de la commande i (i de 1 à 10) :
#   '{"order_id":"O-30%d","customer":"C008","amount":10.0,"city":"Lyon"}' % i
# Envoyez les 10 commandes avec la clé "C008", attendez les confirmations (flush),
# puis rangez lag("billing", "orders") dans lag_after.
lag_after = {}

print("lag :", lag_after)

In [ ]:
# Vérification 6.2 (a)
assert lag_after == {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 10}, "À revoir : lag de 10 sur la partition 5 seulement : %s" % lag_after
print("OK  lag de 10 sur la partition 5 seulement : %s" % lag_after)

**Exemple en ligne de commande** : la même mesure avec l'outil Java.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing

In [ ]:
# TODO 6.2 (b) : relancer la facturation et compter les messages lus
# Relancez consume() avec le groupe billing et rangez la liste des messages dans reread.
reread = []

print(len(reread), "messages relus :", sorted({m.key().decode() for m in reread}))

In [ ]:
# Vérification 6.2 (b)
assert len(reread) == 10 and {m.key().decode() for m in reread} == {"C008"}, "À revoir : 10 commandes de C008 relues"
print("OK  10 commandes de C008 relues")
assert sum(lag("billing", "orders").values()) == 0, "À revoir : lag résorbé"
print("OK  lag résorbé")

**Résultat attendu** : toutes les commandes de `C008` vont dans la partition 5 ; le lag vaut 10 sur cette partition, 0 ailleurs. La relance lit exactement les 10 nouvelles commandes : le groupe reprend à ses offsets validés.

### 6.3 Deux consommateurs dans un groupe

Les consommateurs d'un même groupe se partagent les partitions : chaque partition est lue par un seul membre. La cellule suivante définit `Instance`, une instance d'un service consommateur qui tourne en arrière-plan dans le notebook, comme un terminal de la version Markdown :

- `Instance("analytics")` démarre un consommateur du groupe `analytics`, abonné à `orders` ;
- `instance.partitions` donne ses partitions, mises à jour en continu ;
- `instance.stop()` le ferme proprement : il quitte le groupe ;
- `Instance.stop_all()` arrête toutes les instances lancées.

In [ ]:
class Instance:
    """Une instance d'un service consommateur : un Consumer qui tourne dans un fil d'exécution."""

    launched = []

    def __init__(self, group, topic="orders", **config):
        self.group = group
        self.partitions = []
        self._stop_event = threading.Event()
        self.consumer = Consumer({"bootstrap.servers": BS, "group.id": group,
                                  "auto.offset.reset": "earliest", "log_level": 4, **config})
        self.consumer.subscribe([topic])
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        Instance.launched.append(self)

    def _loop(self):
        while not self._stop_event.is_set():
            # consume() fait aussi vivre l'appartenance au groupe (rééquilibrages, validations)
            self.consumer.consume(100, timeout=0.5)
            self.partitions = sorted(tp.partition for tp in self.consumer.assignment())
        # Départ propre : le coordinateur est prévenu et rééquilibre aussitôt
        self.consumer.close()

    def stop(self):
        self._stop_event.set()
        self._thread.join(15)

    @classmethod
    def stop_all(cls):
        for instance in cls.launched:
            instance.stop()
        cls.launched = []

    def __repr__(self):
        return "Instance(%s, partitions %s)" % (self.group, self.partitions)

In [ ]:
# TODO 6.3 : deux instances du service analytics
#   1. Lancez deux instances du groupe analytics, dans les variables a et b.
#   2. Attendez que chacune ait des partitions : wait_for(lambda: a.partitions and b.partitions, timeout=60).
#   3. Décrivez le groupe : admin.describe_consumer_groups(["analytics"]) renvoie {group: futur}.
#      Le résultat a les attributs state, type, partition_assignor et members ; chaque membre a
#      member_id, client_id et assignment.topic_partitions. Rangez-le dans analytics_group.
#   4. Demandez le retour au début des offsets d'analytics pendant qu'il est actif :
#        admin.alter_consumer_group_offsets([ConsumerGroupTopicPartitions("analytics",
#            [TopicPartition("orders", p, 0) for p in range(6)])])
#      renvoie {group: futur}. Rangez dans reset_errors les codes d'erreur des partitions
#      (tp.error.name() pour chaque tp de result.topic_partitions dont tp.error n'est pas None).
a = b = None
analytics_group = None
reset_errors = []

print(a, b)

In [ ]:
# Vérification 6.3
assert (a is not None and b is not None and sorted(a.partitions + b.partitions) == list(range(6))
         and len(a.partitions) == 3), "À revoir : 3 partitions par instance : %s et %s" % (a, b)
print("OK  3 partitions par instance : %s et %s" % (a, b))
assert (analytics_group is not None and analytics_group.state == ConsumerGroupState.STABLE
         and len(analytics_group.members) == 2), "À revoir : groupe STABLE avec 2 membres"
print("OK  groupe STABLE avec 2 membres")
assert analytics_group.partition_assignor == "range", "À revoir : stratégie range"
print("OK  stratégie range")
assert len(reset_errors) == 6 and set(reset_errors) == {"UNKNOWN_MEMBER_ID"}, "À revoir : repositionnement refusé sur les 6 partitions : %s" % sorted(set(reset_errors))
print("OK  repositionnement refusé sur les 6 partitions : %s" % sorted(set(reset_errors)))

**Vue en ligne de commande**, pendant que les deux instances tournent. Le repositionnement est demandé en simulation (`--dry-run`) : l'outil le refuse aussi.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --members --verbose
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --state
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group analytics --topic orders --reset-offsets --to-earliest --dry-run

**Résultat attendu** : chaque instance reçoit 3 partitions, `[0, 1, 2]` et `[3, 4, 5]`. État `STABLE`, 2 membres, stratégie `range`, client `rdkafka`. Le repositionnement est refusé partition par partition (`UNKNOWN_MEMBER_ID`) ; l'outil Java refuse avant d'envoyer quoi que ce soit : `Assignments can only be reset if the group 'analytics' is inactive, but the current state is Stable.`

### 6.4 Départ d'un consommateur

Quand un membre quitte le groupe, le coordinateur rééquilibre : il redistribue ses partitions aux membres restants.

In [ ]:
# TODO 6.4 : arrêter l'instance b et suivre le groupe
#   1. Notez start = time.time(), puis arrêtez b.
#   2. Toutes les 0,5 s, pendant au plus 20 s, décrivez le groupe analytics et ajoutez à states
#      le couple (secondes depuis start arrondies au dixième, group.state.name).
#      Arrêtez-vous quand l'état est STABLE et que a.partitions compte 6 partitions.
#   3. Arrêtez a, puis rangez dans final_state le nom de l'état du groupe.
states = []
final_state = None

print(states)
print("a :", a, "- état final :", final_state)

In [ ]:
# Vérification 6.4
assert states and states[-1][1] == "STABLE", "À revoir : groupe de nouveau STABLE après le départ de b"
print("OK  groupe de nouveau STABLE après le départ de b")
assert a is not None and a.partitions == list(range(6)), "À revoir : a a reçu les 6 partitions"
print("OK  a a reçu les 6 partitions")
assert final_state == "EMPTY", "À revoir : groupe EMPTY après l'arrêt de a"
print("OK  groupe EMPTY après l'arrêt de a")

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --state

**Résultat attendu** : en général `PREPARING_REBALANCING` pendant une à deux secondes, puis `STABLE` ; l'instance `a` reçoit les 6 partitions. Après son arrêt, le groupe est `EMPTY`, sans membre.

### 6.5 Le nouveau protocole de groupe

Avec le nouveau protocole (KIP-848, `group.protocol=consumer`), le coordinateur calcule lui-même l'affectation cible, et chaque membre converge vers elle sans arrêt global du groupe. Le protocole classique reste la valeur par défaut des clients en Kafka 4.3.

In [ ]:
# TODO 6.5 (a) : le groupe analytics-v2 avec le nouveau protocole
#   1. Lancez deux instances du groupe analytics-v2 avec l'option group.protocol=consumer, dans c et d :
#        Instance("analytics-v2", **{"group.protocol": "consumer"})
#   2. Attendez que les 6 partitions soient réparties entre les deux (wait_for, timeout=60).
#   3. Décrivez le groupe dans group_v2. Chaque membre a aussi un attribut target_assignment.
#   4. admin.list_consumer_groups().result().valid est la liste des groupes : chacun a group_id
#      et type (type.name vaut CLASSIC ou CONSUMER). Rangez {group_id: type.name} dans types.
c = d = None
group_v2 = None
types = {}

print(c, d)
print(types)

In [ ]:
# Vérification 6.5 (a)
assert group_v2 is not None and group_v2.type == ConsumerGroupType.CONSUMER, "À revoir : analytics-v2 de type CONSUMER"
print("OK  analytics-v2 de type CONSUMER")
assert group_v2.partition_assignor == "uniform", "À revoir : affectation calculée par le serveur : uniform"
print("OK  affectation calculée par le serveur : uniform")
assert len(group_v2.members) == 2, "À revoir : 2 membres"
print("OK  2 membres")
assert (types.get("analytics") == "CLASSIC" and types.get("billing") == "CLASSIC"
         and types.get("analytics-v2") == "CONSUMER"), "À revoir : types des groupes : %s" % types
print("OK  types des groupes : %s" % types)

**Vue en ligne de commande**, pendant que les deux instances tournent :

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics-v2 --members --verbose
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list --type

In [ ]:
# TODO 6.5 (b) : arrêter les deux instances, puis rangez dans state_v2 le couple
# (nom de l'état, partition_assignor) du groupe analytics-v2.
state_v2 = (None, None)

print(state_v2)

In [ ]:
# Vérification 6.5 (b)
assert state_v2 == ("EMPTY", "uniform"), "À revoir : analytics-v2 vide, assignateur uniform : %s" % (state_v2,)
print("OK  analytics-v2 vide, assignateur uniform : %s" % (state_v2,))

**Résultat attendu** : `type` `CONSUMER`, assignateur `uniform`, 2 membres dont l'affectation est égale à la cible. Les colonnes `CURRENT-EPOCH` et `TARGET-EPOCH` de l'outil Java sont renseignées. `--list --type` affiche `Consumer` pour `analytics-v2` et `Classic` pour les autres groupes.

### 6.6 Réinitialiser les offsets

Le groupe `billing` est inactif. On peut repositionner ses offsets, en simulant toujours avant d'appliquer.

**Exemple en ligne de commande** : simulation d'un retour au début. `--dry-run` affiche les nouveaux offsets sans rien modifier.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-earliest --dry-run

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-earliest --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-latest --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --shift-by -5 --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders \
  --reset-offsets --to-current --export --dry-run > /tmp/billing-offsets.csv
```

In [ ]:
# TODO 6.6 : repositionner billing, toujours en simulant d'abord
# 1. Écrivez reset_offsets(group, topic, new_offsets, apply=False). new_offsets est un dictionnaire
#    {partition: nouvel offset}. La fonction :
#      - affiche le tableau partition / offset validé / nouvel offset (c'est la simulation) ;
#      - si appliquer vaut True, applique avec admin.alter_consumer_group_offsets(...) (voir 6.3)
#        et lève RuntimeError si une partition porte une erreur.
def reset_offsets(group, topic, new_offsets, apply=False):
    pass


# 2. Retour au début : simulez, puis appliquez topic_offsets("orders", OffsetSpec.earliest()).
#    Rangez le lag total (somme des valeurs de lag()) dans lag_start.
# 3. Placez billing à la fin, puis reculez de 5 messages par partition, sans descendre sous
#    le premier offset disponible. Rangez le lag total dans lag_rewind.
# 4. Exportez les offsets validés dans /tmp/billing-offsets.csv : une ligne "topic,partition,offset"
#    par partition, comme l'export de l'outil Java.
lag_start = lag_rewind = None

print("lag_debut :", lag_start, "- lag_recul :", lag_rewind)

In [ ]:
# Vérification 6.6
assert lag_start == 70, "À revoir : après retour au début, lag égal au nombre de messages : %s" % lag_start
print("OK  après retour au début, lag égal au nombre de messages : %s" % lag_start)
assert lag_rewind == 20, "À revoir : après recul de 5, lag de 20 (5 sur les 4 partitions non vides) : %s" % lag_rewind
print("OK  après recul de 5, lag de 20 (5 sur les 4 partitions non vides) : %s" % lag_rewind)
try:
    lines = open("/tmp/billing-offsets.csv").read().split()
except OSError:
    lines = []
assert len(lines) == 6 and all(l.startswith("orders,") for l in lines), "À revoir : export CSV de 6 lignes"
print("OK  export CSV de 6 lignes")

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing

**Résultat attendu** : après le retour au début, un lag total de 70, tous les messages du topic. Après le recul de 5, un lag de 20 : 5 sur chacune des partitions 0, 1, 2 et 5 ; les partitions vides 3 et 4 restent à 0. Le fichier contient une ligne par partition, de `orders,0,7` à `orders,5,16`.

### 6.7 Défi : retraiter la dernière heure

La facturation a mal traité toutes les commandes reçues depuis une heure. Repositionnez `billing` pour les retraiter, en simulant d'abord.

**Exemple en ligne de commande** : les options de l'outil Java qui parlent de temps.

In [ ]:
!docker exec kafka-tools bash -c "kafka-consumer-groups.sh --help 2>&1 | grep -A2 -E '^--(by-duration|to-datetime)'"

In [ ]:
# TODO 6.7 (défi) : repositionner billing sur les commandes reçues depuis une heure
# OffsetSpec.for_timestamp(ms) donne, pour chaque partition, le premier offset dont l'horodatage
# est postérieur ou égal à ms (millisecondes depuis 1970), ou -1 si aucun message n'est aussi récent.
# time.time() donne l'heure courante en secondes.
# Construisez new_offsets_1h {partition: offset}, simulez avec reset_offsets(), puis appliquez.
new_offsets_1h = {}

print(new_offsets_1h)

In [ ]:
# Vérification 6.7 : comparaison avec la simulation de l'outil Java
output = !docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --by-duration PT1H --dry-run 2>&1
output = output.n
expected_java = {int(p): int(o) for p, o in re.findall(r"^billing\s+orders\s+(\d+)\s+(\d+)", output, re.M)}
print("outil Java :", expected_java)
assert new_offsets_1h and new_offsets_1h == expected_java, "À revoir : mêmes offsets que --by-duration PT1H"
print("OK  mêmes offsets que --by-duration PT1H")
assert committed_offsets("billing", "orders") == new_offsets_1h, "À revoir : repositionnement appliqué"
print("OK  repositionnement appliqué")

**Résultat attendu** : si le TP2 date de moins d'une heure, l'offset 0 partout, comme `--to-earliest`. Si le déjeuner les sépare, seules les commandes de `C008` (étape 6.2) ont moins d'une heure : la partition 5 revient à l'offset 11, les autres partitions sont placées à la fin.

### 6.8 Rétention en temps

La rétention supprime des **segments** entiers, jamais des messages isolés. Un segment est fermé quand il atteint `segment.bytes` ou `segment.ms` : le message suivant ouvre un nouveau segment. Un segment est supprimé quand son message le plus récent dépasse `retention.ms`, y compris le segment actif (celui où le broker écrit) : le broker en ouvre alors un nouveau, vide.

**Exemple en ligne de commande** : la fréquence de vérification de la rétention, réglage de lab.

In [ ]:
!grep -B1 log.retention.check.interval.ms config/kafka/kafka-1/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic audit-short --partitions 1 --replication-factor 3 \
  --config retention.ms=60000 --config segment.ms=10000
seq 1 100 | sed 's/^/audit-/' | kafka-console-producer.sh --bootstrap-server $BS --topic audit-short
sleep 35; echo audit-101 | kafka-console-producer.sh --bootstrap-server $BS --topic audit-short
kafka-get-offsets.sh --bootstrap-server $BS --topic audit-short --time earliest
```

Pourquoi 35 s : plus que `segment.ms` (10 s), pour que `audit-101` ouvre un nouveau segment ; et plus que la période de vérification de la rétention (30 s dans le lab), pour qu'une vérification passe entre l'expiration du premier segment et celle du second.

In [ ]:
# TODO 6.8 : faire supprimer des messages par la rétention
#   1. Créez audit-short : 1 partition, RF 3, avec retention.ms=60000 et segment.ms=10000
#      (NewTopic(..., config={"retention.ms": "60000", ...})).
#   2. Envoyez "audit-1" à "audit-100" (producer), attendez 35 s (time.sleep), puis envoyez "audit-101".
#   3. Attendez que le premier offset disponible de la partition 0,
#      topic_offsets("audit-short", OffsetSpec.earliest())[0], atteigne au moins 100 :
#      wait_for(..., timeout=180, step=5). Rangez la durée d'attente en secondes dans retention_duration.
retention_duration = None

print("duree_retention :", retention_duration)

In [ ]:
# Vérification 6.8
assert retention_duration is not None, "À revoir : suppression du premier segment attendue (%s s)" % retention_duration
print("OK  suppression du premier segment attendue (%s s)" % retention_duration)
first_offset = topic_offsets("audit-short", OffsetSpec.earliest())[0]
assert first_offset in (100, 101), "À revoir : premier offset disponible : %d (100, puis 101 quand le segment actif expire à son tour)" % first_offset
print("OK  premier offset disponible : %d (100, puis 101 quand le segment actif expire à son tour)" % first_offset)
assert end_offsets("audit-short") == {0: 101}, "À revoir : 101 messages écrits"
print("OK  101 messages écrits")

In [ ]:
!docker compose exec -T kafka-1 ls -l /var/lib/kafka/data/audit-short-0

**Résultat attendu** :

- d'abord `{0: 0}` et `{0: 100}` ;
- entre 1 min et 1 min 40 après le 100e message, le premier offset passe à 100. Dans le répertoire de la partition, l'ancien segment est renommé `00000000000000000000.log.deleted`, puis effacé une minute plus tard ; le segment actif est `00000000000000000100.log` ;
- 30 à 60 s plus tard, le premier offset passe à 101 : le segment actif, qui ne contient que `audit-101`, a expiré à son tour. La vérification accepte donc 100 ou 101, selon le moment où vous l'exécutez.

### 6.9 Compaction

Un topic compacté (`cleanup.policy=compact`) garde au moins le dernier message de chaque clé. Le nettoyeur (« log cleaner ») ne travaille que sur les segments fermés.

La fonction fournie ci-dessous lit tous les messages encore présents dans une partition. Avec un topic compacté, les offsets ne se suivent plus : elle lit jusqu'au dernier offset du topic au lieu de compter les messages.

In [ ]:
def read_all(topic, partition=0):
    """Messages encore présents dans une partition : liste de (offset, key, value)."""
    end = end_offsets(topic)[partition]
    consumer = Consumer({"bootstrap.servers": BS, "group.id": "tp3-lecture",
                         "enable.auto.commit": False, "log_level": 4})
    consumer.assign([TopicPartition(topic, partition, OFFSET_BEGINNING)])
    messages = []
    while end and (not messages or messages[-1][0] < end - 1):
        batch = consumer.consume(100, timeout=5)
        if not batch:
            break
        for m in batch:
            if m.error() is None:
                key = m.key().decode() if m.key() is not None else None
                value = m.value().decode() if m.value() is not None else None
                messages.append((m.offset(), key, value))
    consumer.close()
    return messages

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic product-prices --partitions 1 --replication-factor 3 \
  --config cleanup.policy=compact --config segment.ms=10000 \
  --config min.cleanable.dirty.ratio=0.01 --config delete.retention.ms=1000
kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|' < /opt/lab/data/product-prices.txt
sleep 12
echo 'P-500|9.99' | kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|'
```

In [ ]:
# TODO 6.9 : un référentiel de prix compacte
#   1. Créez product-prices : 1 partition, RF 3, avec cleanup.policy=compact, segment.ms=10000,
#      min.cleanable.dirty.ratio=0.01 et delete.retention.ms=1000.
#   2. Envoyez les 20 lignes de "data/product-prices.txt" (format "produit|prix",
#      le produit est la clé), attendez 12 s, puis envoyez la clé "P-500" avec la valeur "9.99".
#   3. Attendez que read_all("product-prices") ne renvoie plus que 5 messages
#      (wait_for, timeout=180, step=5), et rangez la liste finale dans prices.
prices = []

table(prices, ["offset", "clé", "valeur"])

In [ ]:
# Vérification 6.9
assert len(prices) == 5 and sorted(m[1] for m in prices) == ["P-100", "P-200", "P-300", "P-400", "P-500"], "À revoir : un seul prix par produit"
print("OK  un seul prix par produit")
assert prices[0][0] > 0, "À revoir : les offsets conservés ne se suivent pas : premier offset %s" % (prices[0][0] if prices else None)
print("OK  les offsets conservés ne se suivent pas : premier offset %s" % (prices[0][0] if prices else None))

**Résultat attendu** : 21 messages juste après l'envoi de `P-500`, puis 5 en moins d'une minute : le dernier prix de `P-100`, `P-200`, `P-300`, `P-400` et `P-500`, aux offsets 16 à 20.

### 6.10 Tombstone

Pour supprimer une clé d'un topic compacté, on envoie un **tombstone** : un message avec cette clé et une valeur nulle (`value=None` en Python). Le nettoyeur supprime d'abord les anciennes valeurs, puis, plus tard, le tombstone lui-même.

Équivalent en ligne de commande de l'envoi du tombstone (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
echo 'P-500|NULL' | kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|' --reader-property null.marker=NULL
```

In [ ]:
# TODO 6.10 : supprimer P-500 avec un tombstone, et observer le nettoyage
#   1. Envoyez la clé "P-500" avec la valeur None.
#   2. Relevez aussitôt les valeurs lisibles de P-500 :
#        [v for (o, k, v) in read_all("product-prices") if k == "P-500"]
#      et ajoutez à steps le couple (0, values).
#   3. Puis toutes les 15 s, pendant au plus 3 minutes : envoyez un message "P-90i" (i = 1, 2...)
#      pour faire travailler le nettoyeur, relevez les valeurs de P-500 et ajoutez
#      (secondes écoulées, values) à steps. Arrêtez-vous quand plus aucune valeur n'est lisible.
steps = []

for t, values in steps:
    print("t+%3d s : %s" % (t, values))

In [ ]:
# Vérification 6.10
assert steps and steps[0][1] == ["9.99", None], "À revoir : au début, l'ancienne valeur et le tombstone coexistent"
print("OK  au début, l'ancienne valeur et le tombstone coexistent")
assert steps[-1][1] == [], "À revoir : à la fin, plus aucune trace de P-500"
print("OK  à la fin, plus aucune trace de P-500")

**Résultat attendu** :

- d'abord, l'ancienne valeur `9.99` et le tombstone `None` coexistent ;
- ensuite, l'ancienne valeur disparaît, le tombstone reste ;
- enfin, environ une minute après l'envoi, plus aucune valeur pour `P-500`.

## État des groupes

La cellule suivante résume l'état des groupes de consommateurs à la fin du TP.

In [ ]:
admin = AdminClient(CONF)
def group_state(name):
    admin = AdminClient(CONF)
    g = admin.describe_consumer_groups([name])[name].result()
    behind = lag(name, "orders")
    return (name, g.type.name, g.state.name, len(g.members), sum(behind.values()) if behind else "?")


existing = {g.group_id for g in admin.list_consumer_groups().result().valid}
lines = []
for name in ("billing", "analytics", "analytics-v2"):
    if name not in existing:
        lines.append((name, "absent", "", "", ""))
        continue
    lines.append(group_state(name))
table(lines, ["Groupe", "Type", "État", "Membres", "Lag sur orders"])

**Attendu** : les trois groupes `EMPTY`, sans membre. `billing` est `CLASSIC`, avec le lag laissé par l'étape 6.7 ; `analytics` est `CLASSIC` et `analytics-v2` est `CONSUMER`, sans lag. Ne supprimez pas ces groupes : le TP6 les réutilise.

## 7. Questions de réflexion

1. Pourquoi la relance de la facturation en 6.2 lit-elle 10 messages et non 70, alors que `auto.offset.reset` vaut `earliest` ?
2. Que se passerait-il avec 8 instances dans le groupe `analytics` ?
3. Pourquoi Kafka refuse-t-il de repositionner les offsets d'un groupe actif ? Comment procéder en production ?
4. Qu'apporte le nouveau protocole de groupe (KIP-848) lors d'un rééquilibrage ?
5. Pourquoi le message `audit-101` a-t-il survécu aux 100 premiers, alors que la rétention est la même pour tous ?
6. Un consommateur qui lit `product-prices` depuis le début voit-il toujours exactement un message par produit ?
7. Pourquoi un tombstone n'est-il pas supprimé immédiatement ?